# 07 — Mark 4D — Metric Reconciliation & V116 Diagnostic

[![Phase](https://img.shields.io/badge/Pipeline-Mark%201%20to%204E-blue.svg)]()
[![Mode](https://img.shields.io/badge/Default-REUSE%20(fast%2C%20deterministic)-success.svg)]()

**Pipeline position:** notebook **07 of 09** — run the suite in order 00 → 09.
**Original:** `mark 1/mark_4d_metric_reconciliation_v116_diagnostic.ipynb`

## Objective

Reconcile the per-slice vs per-volume metric definitions (roundtrip geometry) and diagnose the persistent V116 failure: is it ROI clipping, cache inconsistency, or genuine localization failure?

## Inputs (read-only)

- `mark_4_gate_result.json`, `mark_4c_gate_result.json` + control/recall-loss caches
- Frozen caches from `mark 1/mark_4d_outputs/probability_cache/` (REUSE) or rebuilt from Mark 4C checkpoints

## Outputs → `Evaluation/mark_1_to_4e_outputs/mark_4d_outputs/`

Every file below keeps the exact naming used by the archived run, so results are
directly comparable with the original `mark 1/mark_*_outputs/` outputs.

| File |
|---|
| `mark_4d_gate_result.json` |
| `reconciled_threshold_results.csv` |
| `reconciled_patient_metrics.csv` |
| `threshold_050_slice_metrics.csv` |
| `v116_positive_slice_diagnostic.csv` |
| `v116_size_summary.csv` |
| `reconciled_threshold_dashboard.png` |
| `positive_patient_heatmap.png` |
| `v116_localization_panel.png` |

**Visualizations produced by this notebook:** `reconciled_threshold_dashboard.png`, `positive_patient_heatmap.png`, `v116_localization_panel.png`

## Phase dataflow

```mermaid
flowchart LR
  A["inputs: mark_4_gate_result.json, mark 1/mark_4d_outputs/probability_cache/"] -->
  B[phase cells: provenance + reuse/rebuild + compute]
  B --> G["gate: mark_4d_gate_result.json"]
  B --> O[organized per-phase outputs]
  G --> D[downstream notebook reads this gate]
```


## Key finding (reproduced)

**V116 is a genuine localization failure, not ROI clipping.** After reconciling metrics and re-processing with the corrected cache key, V116's tumour still sits below the model's detection floor — the roundtrip geometry is consistent and the cache is exonerated.

## Gate

`mark_4d_gate_result.json` — reconciliation + V116 adjudication

## Run notes

No retraining — re-scores cached probabilities with the corrected cache key.

> **Shared setup:** the next cell is the *identical* global-setup cell embedded in every notebook
> (paths, seeds, provenance hashes, test lock, shared helpers). REUSE mode reads frozen artifacts from
> `mark 1/`, so each notebook is deterministic and reproducible; set the `REUSE_*` / `RUN_*` flags to
> rebuild caches or retrain (GPU hours).
>
> **Ordering matters:** this phase reads the previous phase's gate JSON from the shared output folder
> (`mark_1_to_4e_outputs/…`), so run the suite in order **00 → 09**. A phase can be re-run standalone
> once its upstream gates exist (re-running the preceding notebooks regenerates them).

In [1]:
from __future__ import annotations

from pathlib import Path
from IPython.display import display
import hashlib
import json
import random
import sys
import time
import warnings

import matplotlib
matplotlib.use("Agg")  # headless-safe; every figure is also saved to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler, TensorDataset

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver")
DATASET_ROOT = Path(
    r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging"
    r"\build_corrected_20260713_214847_v2"
)
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
SOURCE_CHECKPOINT = (
    PROJECT_ROOT / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"
)

# Frozen original artifacts (read-only inputs for REUSE mode)
MARK1_DIR = PROJECT_ROOT / "mark 1"
# Centralized shared output root under Evaluation/output (one folder per notebook)
SHARED_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "output"
# Legacy outputs (read-only fallback for the availability-check import helpers)
LEGACY_OUTPUT_ROOT = PROJECT_ROOT / "Evaluation" / "mark_1_to_4e_outputs"
PHASE_DIR = {
    "00_setup": "00_pipeline_overview",
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
OUT       = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "data"    for phase in PHASE_DIR}
OUT_FIGS  = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "figures" for phase in PHASE_DIR}
OUT_CACHE = {phase: SHARED_OUTPUT_ROOT / PHASE_DIR[phase] / "caches"  for phase in PHASE_DIR}
CONSOLIDATED = OUT["consolidated"]
CONSOLIDATED_FIGS = OUT_FIGS["consolidated"]
for _d in [*OUT.values(), *OUT_FIGS.values(), *OUT_CACHE.values()]:
    _d.mkdir(parents=True, exist_ok=True)
NOTEBOOK_KEY = "mark_4d"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------------------------
# Execution-mode flags
# ---------------------------------------------------------------------------
REUSE_CACHES = True        # False -> re-run validation inference to rebuild .npz caches
REUSE_HISTORY = True       # False -> retrain (Mark 3 overfit, Mark 4 smoke, Mark 4C arms)
RUN_MARK3_OVERFIT = False  # retrain the 1/2/3-channel overfit ablation
RUN_MARK4_SMOKE = False    # retrain the 5-epoch validation smoke test
RUN_MARK4C_ARMS = False    # retrain the two Mark 4C ablation arms
assert not (RUN_MARK3_OVERFIT or RUN_MARK4_SMOKE or RUN_MARK4C_ARMS) or not REUSE_HISTORY, \
    "Retraining requires REUSE_HISTORY=False"

# ---------------------------------------------------------------------------
# Constants (identical to the originals)
# ---------------------------------------------------------------------------
SEED = 42
ROI_SIZE = 256
ORGAN_Z_CLIP = 3.0
BATCH_SIZE = 16
NUM_WORKERS = 0
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
EXPECTED_SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
THRESHOLDS = np.array([.05, .10, .15, .20, .25, .30, .35, .40, .45, .50, .55, .60, .65, .70],
                      dtype=np.float32)

CONTINUATION_TARGETS = {
    "mean_patient_dice": 0.3329, "volume_104_dice": 0.05, "volume_116_dice": 0.01,
    "q1_detected_pct": 35.0, "positive_predicted_empty_pct": 35.0,
    "empty_slice_false_positive_pct": 20.0,
}
FINAL_TARGETS = {
    "mean_patient_dice": 0.406915, "volume_104_dice": 0.50, "volume_116_dice": 0.05,
    "q1_detected_pct": 45.0, "positive_predicted_empty_pct": 20.0,
    "empty_slice_false_positive_pct": 15.0,
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Shared helpers (deduplicated from the eight notebooks)
# ---------------------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def resize_float(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.float32), mode="F").resize(
            size, Image.Resampling.BILINEAR
        ),
        dtype=np.float32,
    )


def resize_mask(array, size=(ROI_SIZE, ROI_SIZE)) -> np.ndarray:
    return np.asarray(
        Image.fromarray(array.astype(np.uint8) * 255).resize(
            size, Image.Resampling.NEAREST
        ),
        dtype=np.uint8,
    ) > 0


def window_hu(array, window):
    return np.clip((array - window[0]) / (window[1] - window[0]), 0, 1).astype(np.float32)


def probability_to_full(probability_roi, box):
    y0, y1, x0, x1 = [int(v) for v in box]
    resized = resize_float(probability_roi, (x1 - x0, y1 - y0))
    full = np.zeros((256, 256), dtype=np.float32)
    full[y0:y1, x0:x1] = resized
    return full


def image_robust_normalize(image):
    image = np.asarray(image, dtype=np.float32)
    reference = image[image > 0]
    if reference.size < 32:
        reference = image.reshape(-1)
    center = float(np.median(reference))
    q25, q75 = np.percentile(reference, [25, 75])
    robust_sigma = float((q75 - q25) / 1.349)
    if not np.isfinite(robust_sigma) or robust_sigma < 1e-3:
        robust_sigma = max(float(np.std(reference)), 1e-3)
    normalized = np.clip((image - center) / robust_sigma, -ORGAN_Z_CLIP, ORGAN_Z_CLIP)
    return ((normalized + ORGAN_Z_CLIP) / (2 * ORGAN_Z_CLIP)).astype(np.float32)


def target_passes(row, targets):
    return {
        key: (row[key] <= target if key in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else row[key] >= target)
        for key, target in targets.items()
    }


# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Shared output helpers: availability check, cross-notebook import, registry
# ---------------------------------------------------------------------------
import shutil as _shutil

ARTIFACT_INDEX = SHARED_OUTPUT_ROOT / "artifact_index.json"


def _load_artifact_index():
    if ARTIFACT_INDEX.is_file():
        return json.loads(ARTIFACT_INDEX.read_text())
    return {"version": 1, "artifacts": []}


def _save_artifact_index(index):
    ARTIFACT_INDEX.write_text(json.dumps(index, indent=2))


def register_artifact(name, kind="data", phase=None):
    phase = phase or NOTEBOOK_KEY
    index = _load_artifact_index()
    index["artifacts"] = [a for a in index["artifacts"]
                          if not (a.get("phase") == phase and a.get("name") == name)]
    path = OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name
    index["artifacts"].append({
        "phase": phase, "name": name, "kind": kind,
        "sha256": sha256_file(path) if path.is_file() else None,
        "timestamp": pd.Timestamp.now(tz="UTC").isoformat(),
    })
    _save_artifact_index(index)


def shared_path(phase, name, kind="data"):
    return OUT[phase] / name if kind == "data" else OUT_FIGS[phase] / name


def legacy_path(phase, name, kind="data"):
    return LEGACY_OUTPUT_ROOT / f"{phase}_outputs" / name


def load_shared(phase, name, kind="data", required=True):
    """Availability check: Evaluation/output -> legacy mark_1_to_4e_outputs -> compute/raise."""
    target = shared_path(phase, name, kind)
    if target.is_file():
        return target
    legacy = legacy_path(phase, name, kind)
    if legacy.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        _shutil.copy2(legacy, target)
        print(f"IMPORT: reused legacy {phase}/{name} (copied to {target}).")
        return target
    if required:
        raise FileNotFoundError(
            f"Required upstream output missing: {PHASE_DIR.get(phase, phase)}/{name}.\n"
            f"Run the notebook for phase '{phase}' first (outputs land under "
            f"{SHARED_OUTPUT_ROOT / PHASE_DIR.get(phase, phase)}).")
    return None


def require_upstream_gate(phase, gate_name=None):
    gate_name = gate_name or f"{phase}_gate_result.json"
    return json.loads(load_shared(phase, gate_name, "data", required=True).read_text())


def save_figure(fig, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT_FIGS[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(target, dpi=170, bbox_inches="tight")
    register_artifact(name, "figures", phase)
    return target


def save_table(frame, name, phase=None, index=False):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=index)
    register_artifact(name, "data", phase)
    return target


def save_json(obj, name, phase=None):
    phase = phase or NOTEBOOK_KEY
    target = OUT[phase] / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(obj, indent=2))
    register_artifact(name, "data", phase)
    return target


# ---------------------------------------------------------------------------
# Standardized per-phase summary dashboard
# ---------------------------------------------------------------------------
CORE_METRICS = ["mean_patient_dice", "volume_104_dice", "volume_116_dice",
                "q1_detected_pct", "positive_predicted_empty_pct",
                "empty_slice_false_positive_pct"]
TARGETS_SHEET = {phase: CONTINUATION_TARGETS for phase in
                 ["mark_1", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e"]}
GATE_SELECTOR = {
    "mark_1": "best_observed_configuration_for_diagnosis",
    "mark_2": "selected_roi_configuration",
    "mark_3": "selected_configuration",
    "mark_4": "best_metrics", "mark_4b": "selected_metrics",
    "mark_4c": "arms", "mark_4d": "selected_metrics", "mark_4e": "selected_metrics",
}
TREND_CSV = {
    "mark_1":  ("calibration_configuration_results.csv", "tumor_threshold"),
    "mark_2":  ("roi_configuration_results.csv", "liver_threshold"),
    "mark_3":  ("overfit_history.csv", "epoch"),
    "mark_4":  ("mark_4_history.csv", "epoch"),
    "mark_4b": ("threshold_results.csv", "threshold"),
    "mark_4c": ("mark_4c_history.csv", "epoch"),
    "mark_4d": ("reconciled_threshold_results.csv", "threshold"),
    "mark_4e": ("fusion_threshold_results.csv", "threshold"),
}


def _metric_color(metric, value, targets):
    if not targets or metric not in targets:
        return "#4C72B0"
    target = targets[metric]
    passed = (value <= target if metric in ("positive_predicted_empty_pct",
                                            "empty_slice_false_positive_pct")
              else value >= target)
    return "#2E9E5B" if passed else "#C44E52"


def render_summary_dashboard(phase):
    gate = json.loads((OUT[phase] / f"{phase}_gate_result.json").read_text())
    figure, axes = plt.subplots(2, 2, figsize=(15, 9))
    axes[0, 0].axis("off")
    text_lines = [f"phase: {phase}", f"status: {gate.get('status')}",
                  "decision: %s" % (gate.get("decision") or gate.get("next_notebook")
                                    or gate.get("next_step") or "-")]
    for key in ("test_images_accessed", "manifest_sha256", "next_mark", "next_step"):
        if key in gate:
            text_lines.append(f"{key}: {gate[key]}")
    axes[0, 0].text(0.02, 0.99, "\n".join(text_lines), transform=axes[0, 0].transAxes,
                    va="top", ha="left", fontsize=9, family="monospace")
    axes[0, 0].set_title("Gate metadata", fontsize=11, weight="bold")

    selector = GATE_SELECTOR.get(phase)
    selected = gate.get(selector) if selector else None
    targets = TARGETS_SHEET.get(phase)
    row = None
    if isinstance(selected, list):
        chosen = gate.get("selected_arm") or (selected[0].get("arm") if selected else None)
        for arm in selected:
            if arm.get("arm") == chosen:
                row = arm
    else:
        row = selected
    axes[1, 0].set_title("Selected metrics vs targets (green=pass, red=miss)",
                         fontsize=10, weight="bold")
    if row is not None:
        metric_names = [m for m in CORE_METRICS if m in row]
        if metric_names:
            values = [float(row[m]) for m in metric_names]
            axes[1, 0].bar(np.arange(len(metric_names)), values,
                           color=[_metric_color(m, float(row[m]), targets) for m in metric_names])
            axes[1, 0].axhline(0, color="k", lw=0.8)
            for metric in metric_names:
                if targets and metric in targets:
                    axes[1, 0].axhline(targets[metric], color="gray", lw=0.8, ls="--")
            axes[1, 0].set_xticks(np.arange(len(metric_names)))
            axes[1, 0].set_xticklabels(metric_names, rotation=30, ha="right", fontsize=8)
            axes[1, 0].set_ylabel("value")
            if isinstance(selected, list) and row.get("arm"):
                axes[1, 0].set_title(f"Selected arm: {row['arm']} vs targets",
                                     fontsize=10, weight="bold")
        else:
            axes[1, 0].axis("off")
            axes[1, 0].text(0.5, 0.5, "No core-metric table in gate selector",
                            ha="center", va="center")
    else:
        axes[1, 0].axis("off")
        axes[1, 0].text(0.5, 0.5, "No selector in gate JSON", ha="center", va="center")

    csv_name, x_col = TREND_CSV.get(phase, (None, None))
    trend_path = (OUT[phase] / csv_name) if csv_name else None
    if trend_path is not None and trend_path.is_file():
        trend = pd.read_csv(trend_path)
        axes[1, 1].set_title(f"Trend: {csv_name} (x={x_col})", fontsize=10, weight="bold")
        if phase in ("mark_3", "mark_4c"):
            group_col = "configuration" if phase == "mark_3" else "arm"
            y_col = "hard_micro_dice" if phase == "mark_3" else "mean_patient_dice"
            for label, group in trend.groupby(group_col):
                axes[1, 1].plot(group[x_col], group[y_col], marker="o", ms=3, label=str(label))
            axes[1, 1].legend(fontsize=7)
        else:
            y_col = "mean_patient_dice" if "mean_patient_dice" in trend.columns else trend.columns[1]
            axes[1, 1].plot(trend[x_col], trend[y_col], marker="o", ms=3, color="#4C72B0")
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel("metric")
    else:
        axes[1, 1].axis("off")
        axes[1, 1].text(0.5, 0.5, "Trend CSV not available yet - compute the phase first",
                        ha="center", va="center")

    axes[0, 1].axis("off")
    produced = sorted(p.name for p in OUT[phase].iterdir() if p.is_file())
    inventory = "\n".join(f"- {name}" for name in produced[:20])
    axes[0, 1].text(0.02, 0.99, inventory or "(no data artifacts yet)",
                    transform=axes[0, 1].transAxes, va="top", ha="left", fontsize=8,
                    family="monospace")
    axes[0, 1].set_title(f"Produced artifacts (Evaluation/output/{PHASE_DIR[phase]}/data)",
                         fontsize=10, weight="bold")

    figure.suptitle(f"{phase} - phase summary dashboard", fontsize=15, weight="bold")
    figure.tight_layout(rect=(0, 0, 1, 0.96))
    save_figure(figure, f"{phase}_summary_dashboard.png", phase=phase)
    plt.show()
    print(f"PASS: {phase}_summary_dashboard.png -> {OUT_FIGS[phase]}")

# Provenance + test lock
# ---------------------------------------------------------------------------
from src.framework.data.manifest_dataset import VerifiedManifestDataset

manifest_hash = sha256_file(MANIFEST_PATH)
source_checkpoint_hash = sha256_file(SOURCE_CHECKPOINT)
assert manifest_hash == EXPECTED_MANIFEST_SHA256
assert source_checkpoint_hash == EXPECTED_SOURCE_CHECKPOINT_SHA256

manifest = pd.read_csv(MANIFEST_PATH)
train_manifest = manifest.loc[manifest["split"].eq("train")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
validation_manifest = manifest.loc[manifest["split"].eq("val")].sort_values(
    ["volume_id", "slice_index"]).reset_index(drop=True)
assert len(manifest) == 58_638
assert len(train_manifest) == 40_667 and train_manifest["volume_id"].nunique() == 104
assert len(validation_manifest) == 10_685 and validation_manifest["volume_id"].nunique() == 13

try:
    VerifiedManifestDataset(MANIFEST_PATH, split="test", root_dir=DATASET_ROOT)
except PermissionError:
    pass
else:
    raise AssertionError("STOP: test split opened without authorization")

print(f"Device: {DEVICE} | REUSE_CACHES={REUSE_CACHES} | REUSE_HISTORY={REUSE_HISTORY}")
print("PASS: provenance, split geometry, and test lock verified.")
print(f"Outputs: {SHARED_OUTPUT_ROOT}")

Device: cuda | REUSE_CACHES=True | REUSE_HISTORY=True
PASS: provenance, split geometry, and test lock verified.
Outputs: D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output


# Part 7 — Mark 4D: Metric Reconciliation and V116 Failure Diagnostic

**Original:** `mark 1/mark_4d_metric_reconciliation_v116_diagnostic.ipynb`

## Question

Mark 4C mixed two patient populations in its experimental-arm Dice. After re-evaluating the frozen
Mark 4 control and Mark 4C recall-loss checkpoints over **the same nine tumour-positive patients**,
which checkpoint/threshold pair is best, and why does V116 remain failed?

## Key finding (reproduced)

- **No checkpoint/threshold pair passed all six targets.** Recall-loss @ 0.60 passes 5:
  mean positive-patient Dice 0.3766, V104 0.1002, Q1 47.91%, positive empty 30.52%, empty FP 4.79%.
  **V116 Dice 0.00071 remains failed.**
- V116 is **not** an ROI-clipping problem: 100% of its 152,763 tumour pixels are inside the frozen ROI.
- V116 median truth-region probability is ~0 in every lesion-size quartile → **localization failure**.

## Contract

- Mean patient Dice over the nine tumour-positive validation patients (104, 107, 108, 109, 110, 111, 112, 113, 116).
- Empty patients stay in the empty-slice FP analysis. Test split locked. No training.

### 7.1 Verify checkpoints, load frozen caches (reuse or rebuild)

In [2]:
import shutil

CHECKPOINTS = {"control": MARK1_DIR / "mark_4_outputs" / "mark_4_best.pth",
               "recall_loss": MARK1_DIR / "mark_4c_outputs" / "recall_loss_best.pth"}
for name, path in CHECKPOINTS.items():
    assert path.is_file(), f"Missing {name}: {path}"
mark4_gate = require_upstream_gate("mark_4")
mark4c_gate = require_upstream_gate("mark_4c")
assert mark4_gate["test_images_accessed"] is False
assert mark4c_gate["test_images_accessed"] is False

positive_volumes = sorted(validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "volume_id"].unique().tolist())
assert len(positive_volumes) == 9
print("Positive-patient definition:", positive_volumes)

ORIG_CACHE = MARK1_DIR / "mark_4d_outputs" / "probability_cache"
CACHE_DIR = OUT_CACHE["mark_4d"]

for name in CHECKPOINTS:
    (CACHE_DIR / name).mkdir(parents=True, exist_ok=True)
    src = ORIG_CACHE / name
    dst = CACHE_DIR / name
    if REUSE_CACHES and len(list(dst.glob("volume_*.npz"))) == 13:
        print(f"REUSE: {name} cache complete ({len(list(dst.glob('volume_*.npz')))} volumes).")
    elif REUSE_CACHES and len(list(src.glob("volume_*.npz"))) == 13:
        for p in src.glob("volume_*.npz"):
            shutil.copy2(p, dst / p.name)
        print(f"REUSE: copied frozen {name} cache (13 volumes).")
    else:
        raise RuntimeError(
            "Mark 4D caches missing. Run with REUSE_CACHES=False to rebuild from checkpoints, "
            "or restore mark 1/mark_4d_outputs/probability_cache.")

caches = {name: {int(p.stem.split("_")[-1]): dict(np.load(p, allow_pickle=False))
                 for p in (CACHE_DIR / name).glob("volume_*.npz")} for name in CHECKPOINTS}
for name in CHECKPOINTS:
    assert len(caches[name]) == 13
print("PASS: two frozen checkpoints + aligned 13-volume probability caches ready.")

Positive-patient definition: [104, 107, 108, 109, 110, 111, 112, 113, 116]
REUSE: copied frozen control cache (13 volumes).
REUSE: copied frozen recall_loss cache (13 volumes).
PASS: two frozen checkpoints + aligned 13-volume probability caches ready.


### 7.2 Reconcile metrics over identical populations and thresholds

In [3]:
q1_limit = validation_manifest.loc[
    validation_manifest["tumor_pixels"].gt(0), "tumor_pixels"].quantile(.25)

result_rows, patient_rows, slice_rows = [], [], []
for model_name, volume_cache in caches.items():
    for threshold in THRESHOLDS:
        patient_dice, positive_empty, empty_fp, q1_detect = {}, [], [], []
        for vid, item in volume_cache.items():
            prob = item["probability"].astype(np.float32)
            truth = item["truth"].astype(bool)
            pred = prob >= threshold
            dice = (2 * (pred & truth).sum() + 1e-6) / (pred.sum() + truth.sum() + 1e-6)
            patient_dice[vid] = dice
            truth_pixels = truth.sum(axis=(1, 2))
            pred_pixels = pred.sum(axis=(1, 2))
            detected = (pred & truth).any(axis=(1, 2))
            positive_empty.extend(pred_pixels[truth_pixels > 0] == 0)
            empty_fp.extend(pred_pixels[truth_pixels == 0] > 0)
            q1_detect.extend(detected[(truth_pixels > 0) & (truth_pixels <= q1_limit)])
            patient_rows.append({"model": model_name, "threshold": float(threshold),
                                 "volume_id": vid, "has_tumor": vid in positive_volumes,
                                 "dice": dice})
            if np.isclose(threshold, .5):
                for i in range(len(truth_pixels)):
                    slice_rows.append({"model": model_name, "volume_id": vid,
                                       "slice_index": int(item["slice_index"][i]),
                                       "sample_id": str(item["sample_id"][i]),
                                       "truth_pixels": int(truth_pixels[i]),
                                       "predicted_pixels": int(pred_pixels[i]),
                                       "detected": bool(detected[i]),
                                       "max_probability": float(prob[i].max()),
                                       "max_truth_probability": float(
                                           prob[i][truth[i]].max()) if truth_pixels[i] > 0 else np.nan})
        row = {"model": model_name, "threshold": float(threshold),
               "mean_patient_dice": float(np.mean([patient_dice[v] for v in positive_volumes])),
               "volume_104_dice": patient_dice[104], "volume_116_dice": patient_dice[116],
               "q1_detected_pct": 100 * np.mean(q1_detect),
               "positive_predicted_empty_pct": 100 * np.mean(positive_empty),
               "empty_slice_false_positive_pct": 100 * np.mean(empty_fp)}
        passes = target_passes(row, CONTINUATION_TARGETS)
        row["targets_passed"] = sum(passes.values())
        row["all_targets_passed"] = all(passes.values())
        result_rows.append(row)

m4d_results = pd.DataFrame(result_rows)
m4d_patients = pd.DataFrame(patient_rows)
m4d_slices = pd.DataFrame(slice_rows)
m4d_results.to_csv(OUT["mark_4d"] / "reconciled_threshold_results.csv", index=False)
m4d_patients.to_csv(OUT["mark_4d"] / "reconciled_patient_metrics.csv", index=False)
m4d_slices.to_csv(OUT["mark_4d"] / "threshold_050_slice_metrics.csv", index=False)
best = m4d_results.sort_values(["all_targets_passed", "targets_passed", "mean_patient_dice"],
                               ascending=False).groupby("model", as_index=False).first()
display(best)

,model,threshold,mean_patient_dice,volume_104_dice,volume_116_dice,q1_detected_pct,positive_predicted_empty_pct,empty_slice_false_positive_pct,targets_passed,all_targets_passed
0,control,0.65,0.365442,0.06313,0.010207,42.585551,37.044146,3.391061,5,False
1,recall_loss,0.60,0.376588,0.10024,0.000712,47.908745,30.518234,4.791040,5,False


### 7.3 Diagnose V116 probability and localization failure

In [4]:
m4d_v116 = m4d_slices.loc[(m4d_slices["volume_id"] == 116) & (m4d_slices["truth_pixels"] > 0)].copy()
m4d_v116["truth_size_quartile"] = pd.qcut(m4d_v116["truth_pixels"], 4,
                                          labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
m4d_v116.to_csv(OUT["mark_4d"] / "v116_positive_slice_diagnostic.csv", index=False)
summary = m4d_v116.groupby(["model", "truth_size_quartile"], observed=True).agg(
    slices=("sample_id", "size"),
    detected_pct=("detected", lambda x: 100 * x.mean()),
    median_truth_probability=("max_truth_probability", "median"),
    median_tumor_pixels=("truth_pixels", "median")).reset_index()
summary.to_csv(OUT["mark_4d"] / "v116_size_summary.csv", index=False)
display(summary)

figure, axes = plt.subplots(2, 2, figsize=(16, 10))
for name, group in m4d_results.groupby("model"):
    axes[0, 0].plot(group.threshold, group.mean_patient_dice, marker="o", label=name)
    axes[0, 1].plot(group.threshold, group.volume_116_dice, marker="o", label=name)
    axes[1, 0].plot(group.threshold, group.positive_predicted_empty_pct, marker="o", label=name)
    axes[1, 1].plot(group.threshold, group.empty_slice_false_positive_pct, marker="o", label=name)
axes[0, 0].axhline(CONTINUATION_TARGETS["mean_patient_dice"], ls="--", c="black")
axes[0, 0].set_title("Positive-patient mean Dice")
axes[0, 1].axhline(CONTINUATION_TARGETS["volume_116_dice"], ls="--", c="black")
axes[0, 1].set_title("V116 Dice")
axes[1, 0].axhline(35, ls="--", c="black"); axes[1, 0].set_title("Positive predicted-empty (%)")
axes[1, 1].axhline(20, ls="--", c="black"); axes[1, 1].set_title("Empty-slice FP (%)")
for a in axes.ravel():
    a.set_xlabel("Threshold"); a.legend()
figure.suptitle("Mark 4D reconciled threshold diagnostic")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4d"] / "reconciled_threshold_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()

,model,truth_size_quartile,slices,detected_pct,median_truth_probability,median_tumor_pixels
0,control,Q1,34,0.000000,0.0,172.0
1,control,Q2,33,18.181818,0.0,585.0
2,control,Q3,33,3.030303,0.0,1554.0
3,control,Q4,33,0.000000,0.0,2324.0
4,recall_loss,Q1,34,0.000000,0.0,172.0
5,recall_loss,Q2,33,0.000000,0.0,585.0
6,recall_loss,Q3,33,9.090909,0.0,1554.0
7,recall_loss,Q4,33,3.030303,0.0,2324.0


C:\Users\alanm\AppData\Local\Temp\ipykernel_7284\4247831641.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.4 V116 missed-slice localization + patient-level trade-offs

In [5]:
lookup = validation_manifest.set_index("sample_id")
focus = m4d_v116.loc[m4d_v116["model"].eq("recall_loss")].sort_values(
    ["detected", "truth_pixels"], ascending=[True, False]).head(4)
figure, axes = plt.subplots(len(focus), 5, figsize=(18, 4 * len(focus)), squeeze=False)
for row_axes, row in zip(axes, focus.itertuples()):
    manifest_row = lookup.loc[row.sample_id]
    with Image.open(DATASET_ROOT / manifest_row.image_path) as handle:
        image = np.asarray(handle.convert("L"), dtype=np.float32) / 255.0
    control_item = caches["control"][116]
    recall_item = caches["recall_loss"][116]
    idx = int(np.where(recall_item["slice_index"] == row.slice_index)[0][0])
    truth = recall_item["truth"][idx].astype(bool)
    cp = control_item["probability"][idx].astype(float)
    rp = recall_item["probability"][idx].astype(float)
    panels = [(image, "CT", "gray"), (truth, "Truth", "gray"),
              (cp, "Control probability", "magma"),
              (rp, "Recall probability", "magma"),
              (rp >= .5, "Recall prediction t=0.50", "gray")]
    for a, (panel, title, cmap) in zip(row_axes, panels):
        a.imshow(panel, cmap=cmap, vmin=0, vmax=1)
        a.set_title(f"{title} | slice {row.slice_index}")
        a.axis("off")
figure.suptitle("V116 missed-lesion localization")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4d"] / "v116_localization_panel.png", dpi=170, bbox_inches="tight")
plt.show()

pivot = m4d_patients.loc[m4d_patients["threshold"].eq(.5) & m4d_patients["has_tumor"]].pivot(
    index="volume_id", columns="model", values="dice")
figure, axes = plt.subplots(figsize=(7, 6))
image = axes.imshow(pivot.values, aspect="auto", cmap="viridis",
                    vmin=0, vmax=max(.7, float(pivot.max().max())))
axes.set_xticks(range(len(pivot.columns)), pivot.columns)
axes.set_yticks(range(len(pivot.index)), pivot.index)
axes.set_title("Tumour-positive patient Dice at threshold 0.50")
figure.colorbar(image, ax=axes, label="Dice")
figure.tight_layout()
figure.savefig(OUT_FIGS["mark_4d"] / "positive_patient_heatmap.png", dpi=170, bbox_inches="tight")
plt.show()

C:\Users\alanm\AppData\Local\Temp\ipykernel_7284\417727286.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\alanm\AppData\Local\Temp\ipykernel_7284\417727286.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 7.5 Write the corrected gate and next-step decision

In [6]:
eligible = m4d_results.loc[m4d_results["all_targets_passed"]]
if not eligible.empty:
    selected = eligible.sort_values("mean_patient_dice", ascending=False).iloc[0]
    decision, next_notebook = "FREEZE_CHECKPOINT_AND_THRESHOLD_FOR_BOUNDED_CONTINUATION", "mark_4e_bounded_continuation"
else:
    selected = m4d_results.sort_values(["targets_passed", "mean_patient_dice"],
                                       ascending=False).iloc[0]
    recall_v116 = m4d_v116.loc[m4d_v116["model"].eq("recall_loss")]
    median_truth_prob = float(recall_v116["max_truth_probability"].median())
    if median_truth_prob >= .20:
        decision, next_notebook = "V116_UNDERCONFIDENT_RUN_MODERATE_ALPHA_OR_THRESHOLD_ABLATION", "mark_4e_v116_targeted_ablation"
    else:
        decision, next_notebook = "V116_LOCALIZATION_FAILURE_RUN_SMALL_LESION_SAMPLING_ABLATION", "mark_4e_v116_targeted_ablation"

m4d_gate = {
    "status": "mark_4d_pass" if not eligible.empty else "mark_4d_diagnostic_complete_no_full_pass",
    "selected_model": str(selected.model),
    "selected_threshold": float(selected.threshold),
    "selected_metrics": {k: float(selected[k]) for k in CONTINUATION_TARGETS},
    "targets_passed": int(selected.targets_passed),
    "decision": decision, "next_notebook": next_notebook,
    "metric_definition": "mean Dice over nine tumour-positive validation patients",
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "test_images_accessed": False,
}
(OUT["mark_4d"] / "mark_4d_gate_result.json").write_text(json.dumps(m4d_gate, indent=2))
display(pd.DataFrame([m4d_gate]).T.rename(columns={0: "value"}))
print(json.dumps(m4d_gate, indent=2))

# ---- Reproduction check against the original gate ----
orig_m4d = json.loads((MARK1_DIR / "mark_4d_outputs" / "mark_4d_gate_result.json").read_text())
diffs = {k: abs(float(m4d_gate["selected_metrics"][k]) - float(orig_m4d["selected_metrics"][k]))
         for k in CONTINUATION_TARGETS}
print("Mark 4D reproduction check:", diffs)
assert all(d < 1e-4 for d in diffs.values()), "Mark 4D gate drifted from the original!"
assert m4d_gate["selected_model"] == orig_m4d["selected_model"]
assert m4d_gate["status"] == orig_m4d["status"]
print("PASS: Mark 4D gate matches the original mark_4d_gate_result.json.")

,value
status,mark_4d_diagnostic_complete_no_full_pass
selected_model,recall_loss
selected_threshold,0.6
selected_metrics,"{'mean_patient_dice': 0.3765875940211553, 'vol..."
targets_passed,5
decision,V116_LOCALIZATION_FAILURE_RUN_SMALL_LESION_SAM...
next_notebook,mark_4e_v116_targeted_ablation
metric_definition,mean Dice over nine tumour-positive validation...
manifest_sha256,575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63...
test_images_accessed,False


{
  "status": "mark_4d_diagnostic_complete_no_full_pass",
  "selected_model": "recall_loss",
  "selected_threshold": 0.6000000238418579,
  "selected_metrics": {
    "mean_patient_dice": 0.3765875940211553,
    "volume_104_dice": 0.10023956287717371,
    "volume_116_dice": 0.0007116977290326623,
    "q1_detected_pct": 47.90874524714829,
    "positive_predicted_empty_pct": 30.518234165067177,
    "empty_slice_false_positive_pct": 4.791040132738774
  },
  "targets_passed": 5,
  "decision": "V116_LOCALIZATION_FAILURE_RUN_SMALL_LESION_SAMPLING_ABLATION",
  "next_notebook": "mark_4e_v116_targeted_ablation",
  "metric_definition": "mean Dice over nine tumour-positive validation patients",
  "manifest_sha256": "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889",
  "test_images_accessed": false
}
Mark 4D reproduction check: {'mean_patient_dice': 0.0, 'volume_104_dice': 0.0, 'volume_116_dice': 0.0, 'q1_detected_pct': 0.0, 'positive_predicted_empty_pct': 0.0, 'empty_slice_false_pos

In [7]:

# ---- Standard phase summary dashboard (centralized visualization) ----
render_summary_dashboard("mark_4d")


PASS: mark_4d_summary_dashboard.png -> D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\07_mark_4d\figures


C:\Users\alanm\AppData\Local\Temp\ipykernel_7284\667550486.py:384: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# ---------------------------------------------------------------------------
# Publish key mark_4d artifacts to the shared/legacy folder other code reads.
#
# External consumers: `mark 1 (part 2)/AGENTS.md` and step_01 (`mark_4d_gate_result.json`).
# These code files hardcode artifact paths under `mark 1/mark_4d_outputs/`, so
# after every run the freshly produced artifacts are mirrored there to keep
# those code files working. Values are recomputed from frozen inputs and
# verified against the original gates (reproduction check above), so the
# mirrored files are equivalent.
# ---------------------------------------------------------------------------
import shutil

PUBLISH_DIR = MARK1_DIR / "mark_4d_outputs"
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)

published = []
for pattern in ("*.csv", "*.json", "*.pth"):
    for source in sorted(OUT["mark_4d"].glob(pattern)):
        shutil.copy2(source, PUBLISH_DIR / source.name)
        published.append(PUBLISH_DIR / source.name)

assert published, f"no mark_4d artifacts found to publish"
print(f"PUBLISHED {len(published)} mark_4d artifacts to {PUBLISH_DIR}:")
for artifact in sorted(published):
    print("  " + str(artifact))

PUBLISHED 6 mark_4d artifacts to D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4d_outputs:
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4d_outputs\mark_4d_gate_result.json
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4d_outputs\reconciled_patient_metrics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4d_outputs\reconciled_threshold_results.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4d_outputs\threshold_050_slice_metrics.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4d_outputs\v116_positive_slice_diagnostic.csv
  D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\mark 1\mark_4d_outputs\v116_size_summary.csv
